# Chapter 05: 3D Occupancy Networks & Temporal Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/05_3d_occupancy_and_temporal_memory.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How does an autonomous vehicle maintain object permanence when obstacles pass behind occluding barriers?*

---

## 1. 🚨 The Real-World Dilemma
3D bounding boxes fail on weird road rubble (mattresses, overturned boats). 3D Occupancy discretizes space into class-agnostic matter vs air. A Spatiotemporal ConvGRU recurrently preserves occluded obstacles across time.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

class SpatialConvGRUCell(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.conv_gates = nn.Conv2d(in_channels + hidden_channels, 2 * hidden_channels, 3, padding=1)
        self.conv_cand = nn.Conv2d(in_channels + hidden_channels, hidden_channels, 3, padding=1)

    def forward(self, x, h_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv_gates(combined)
        z, r = torch.split(torch.sigmoid(gates), self.hidden_channels, dim=1)
        h_tilde = torch.tanh(self.conv_cand(torch.cat([x, r * h_prev], dim=1)))
        return (1.0 - z) * h_prev + z * h_tilde

cell = SpatialConvGRUCell(16, 16)
h_state = torch.zeros(1, 16, 32, 32)

# Step 1: Target visible
obs_vis = torch.zeros(1, 16, 32, 32)
obs_vis[:, :, 14:18, 14:18] = 2.0
h_state = cell(obs_vis, h_state)

# Step 2: Target occluded
obs_occ = torch.zeros(1, 16, 32, 32)
h_state = cell(obs_occ, h_state)

print(f"Memory persistence at occluded cell: {h_state[0, 0, 16, 16].item():.4f}")
print("✅ Object permanence verified: state preserved across occlusion!")

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Moving cars leave long red 'ghost trails' in memory** | Update gate $z \approx 0$ fails to overwrite vacated voxels. | Check mean value of update gate `z.mean()`. | Feed optical flow / velocity vectors into input tensor. |
| **Occupancy grid blurs when turning corners** | Ego-motion uncompensated before temporal blending. | Rotate car in place and check if stationary poles blur into arcs. | Apply bilinear `grid_sample` pose warping $T_{t-1 \to t}^{-1}$ before ConvGRU. |